# MuScripter Flash Training (Colab)

Run long Bach10 / URMP / MusicNet fine-tuning with the training state stored on Google Drive. If Colab disconnects, rerun the notebook and it resumes from the saved state.

The current bundled Bach10 checkpoint remains the base model. A new multidataset checkpoint is only promoted after held-out evaluation improves.

In [ ]:
%pip -q install --upgrade "git+https://github.com/sakusdev/muscriptor.git@main" scipy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Configure dataset locations
Set unavailable datasets to `None`. MusicNet should point at the extracted folder containing `train_data`, `train_labels`, `test_data`, and `test_labels`. URMP should point at the extracted URMP root.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/MuScripterFlash')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

BACH10 = None  # e.g. Path('/content/drive/MyDrive/datasets/Bach10_v1.1')
URMP = None    # e.g. Path('/content/drive/MyDrive/datasets/URMP')
MUSICNET = None  # e.g. Path('/content/drive/MyDrive/datasets/musicnet')

BASE = Path('/content/muscriptor-base.pt')
OUTPUT = DRIVE_ROOT / 'flash-neural-multidataset.pt'
STATE = DRIVE_ROOT / 'flash-training-state.pt'
METRICS = DRIVE_ROOT / 'flash-neural-multidataset-metrics.json'
TARGET_STEPS = 3000
SAVE_EVERY = 50


In [ ]:
import importlib.resources as resources
import shutil

checkpoint = resources.files('muscriptor') / 'checkpoints' / 'flash-neural-bach10.pt'
with resources.as_file(checkpoint) as source:
    shutil.copy2(source, BASE)
print('Base checkpoint:', BASE)
print('Resume state:', STATE if STATE.exists() else 'none (fresh run)')


In [ ]:
import subprocess

cmd = [
    'muscriptor-flash-train',
    '--base', str(BASE),
    '--output', str(OUTPUT),
    '--metrics', str(METRICS),
    '--steps', str(TARGET_STEPS),
    '--state-output', str(STATE),
    '--save-every', str(SAVE_EVERY),
]
for flag, path in [('--bach10', BACH10), ('--urmp', URMP), ('--musicnet', MUSICNET)]:
    if path is not None:
        cmd += [flag, str(path)]
if STATE.exists():
    cmd += ['--resume-state', str(STATE)]

if not any(x in cmd for x in ['--bach10', '--urmp', '--musicnet']):
    raise RuntimeError('Configure at least one dataset path first.')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## Resume after a disconnect
Reconnect Colab, rerun the cells above, and keep the same dataset paths / training parameters. The notebook detects `flash-training-state.pt` on Drive and resumes automatically. `TARGET_STEPS` is the final target step, not an additional number of steps.